In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class PMT:
    def __init__(self, position, r, a):
        self.position = np.array(position)
        self.r = r
        self.a = a

    def get_photoelectrons(self, num_photons):
        if num_photons == 0: return 0
        return np.random.poisson(self.a * num_photons)

class Detector:
    def __init__(self, R=2.0, N=40, r=0.1, a=0.5):
        self.R, self.N, self.r, self.a = R, N, r, a
        self.pmts = self._generate_pmt_fibonacci()
        self.pmt_positions_array = np.array([pmt.position for pmt in self.pmts])

    def _generate_pmt_fibonacci(self):
        pmts = []
        indices = np.arange(0, self.N, dtype=float) + 0.5
        phi = np.arccos(1 - 2 * indices / self.N)
        theta = np.pi * (1 + 5**0.5) * indices
        x = np.cos(theta) * np.sin(phi) * self.R
        y = np.sin(theta) * np.sin(phi) * self.R
        z = np.cos(phi) * self.R
        for i in range(self.N):
            pmts.append(PMT([x[i], y[i], z[i]], self.r, self.a))
        return pmts

    def simulate_event(self, E, source_z=0.0, attenuation_length=None, Y=10000):
        N_photons = np.random.poisson(Y * E)
        if N_photons == 0: return 0

        source_pos = np.array([0.0, 0.0, source_z])
        
        directions = np.random.normal(size=(N_photons, 3))
        directions /= np.linalg.norm(directions, axis=1, keepdims=True)
        B = 2 * np.dot(directions, source_pos)
        C = np.dot(source_pos, source_pos) - self.R**2
        t = (-B + np.sqrt(B**2 - 4*C)) / 2

        hit_points = source_pos + t[:, np.newaxis] * directions
        

        if attenuation_length is not None:
            survival_probs = np.exp(-t / attenuation_length)
            survived_mask = np.random.random(size=N_photons) < survival_probs
            hit_points = hit_points[survived_mask]
            
        if len(hit_points) == 0: return 0

        # 5. Векторный поиск попаданий в ФЭУ 
        diffs = hit_points[:, np.newaxis, :] - self.pmt_positions_array[np.newaxis, :, :]
        dists = np.linalg.norm(diffs, axis=2) 
        

        hits = np.sum(dists <= self.r, axis=0) 

        total_charge = np.sum(np.random.poisson(self.a * hits))
        return total_charge

np.random.seed(2)
detector = Detector(R=2.0, N=40, r=0.1, a=0.5)

z_positions = np.linspace(-1.8, 1.8, 10) # Сдвигаем источник от -1.8 до 1.8 м
mean_charge_no_att = []
mean_charge_with_att = []
num_events = 500 

for z in z_positions:
    q_no_att = [detector.simulate_event(E=1.0, source_z=z, attenuation_length=None) for _ in range(num_events)]
    mean_charge_no_att.append(np.mean(q_no_att))
    
    q_with_att = [detector.simulate_event(E=1.0, source_z=z, attenuation_length=8.0) for _ in range(num_events)]
    mean_charge_with_att.append(np.mean(q_with_att))


plt.figure(figsize=(10, 6))

plt.plot(z_positions, mean_charge_no_att, 'o-', label='Без затухания ', color='blue')
plt.plot(z_positions, mean_charge_with_att, 's-', label='С затуханием ', color='red')

plt.title('Энергетический отклик детектора (E = 1 МэВ) вдоль оси Z')
plt.xlabel('Координата источника по оси Z (м)')
plt.ylabel('Средний собранный заряд (фотоэлектроны)')
plt.grid(True)
plt.legend()
plt.show()